# W3-E — tables and figures

**No GPU.** This notebook reads every run it can find and produces the paper's
tables and figures in minutes.

Run rows are collected from `/kaggle/working` **and** from anything mounted under
`/kaggle/input`, so if you saved W3-A''s outputs as datasets they are picked up
automatically; if you did not, it still reports everything currently in the
session.

## What it computes

**Recording-level bootstrap confidence intervals.** Test clips are not
independent: one recording contributes dozens of clips sharing a channel, a
background-noise field and an operating condition. Resampling clips would treat
fifty clips from one ship as fifty independent pieces of evidence and report an
interval that is far too narrow. We resample **recordings** with replacement,
1,000 times.

**The leak decomposition.** For each leaky protocol the test set is split into
clips whose ship was also seen in training and clips whose was not, and the
metric is reported on each part. This is the mechanism behind the headline gap:
it shows how much of a 99% number is recognition and how much is recall of a ship
the model already met.

**Everything else**: the inflation gap, per-class accuracy, per-seed spread (the
test-set size is a random variable —6,442 to 8,531 clips across the five
vessel-disjoint seeds), the parameter-count ladder, the domain-generalisation
table with its negative control, cross-corpus results, and the label-shuffling
control.

In [ ]:
# --------------------------------------------------------------------------- #
# 1. locate the feature dataset
#
# Kaggle mounts a dataset by slug, in one of two layouts, and it expands
# archives on ingest. So rather than hard-coding a path we look for the one
# thing that must exist: <root>/oceanship/index.csv. If detection ever fails,
# copy the real path out of the listing below into MANUAL and re-run.
# --------------------------------------------------------------------------- #
import csv, json, os, shutil, subprocess, sys, time
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
RUNS = WORK / "w3-runs"
SCRIPTS = WORK / "scripts"
MANUAL = {}          # e.g. MANUAL["features"] = "/kaggle/input/under-water-features/uwatr-features"

T0 = time.time()
def tick(msg):
    print(f"  [{time.time() - T0:7.1f}s] {msg}", flush=True)


def candidates():
    out = []
    for depth in range(1, 5):
        for p in sorted(INPUT.glob("/".join(["*"] * depth))):
            if p.is_dir() and (p / "oceanship" / "index.csv").exists():
                out.append(p)
    return sorted(set(out), key=lambda p: len(p.parts))


MANUAL_FEATURES = MANUAL.get("features")
FEAT = Path(MANUAL_FEATURES) if MANUAL_FEATURES else None
if FEAT is None:
    cands = candidates()
    FEAT = cands[0] if cands else None

print("=" * 88)
print("inputs")
print("=" * 88)
for depth in range(1, 3):
    for p in sorted(INPUT.glob("/".join(["*"] * depth))):
        if p.is_dir():
            kids = sorted(x.name for x in p.iterdir())[:6]
            print(f"  {p}")
            print(f"      {kids}")

if FEAT is None:
    print()
    print("  no feature tree found. A mounted dataset must contain")
    print("  <root>/oceanship/index.csv, <root>/deepship/index.csv, <root>/shipsear/index.csv,")
    print("  /splits. Copy the correct path into MANUAL at the top of this cell and re-run.")
    raise SystemExit("under_water_features not found")

print()
print(f"  features root : {FEAT}")
need = [FEAT / c / "index.csv" for c in ("oceanship", "deepship", "shipsear")]
need += [FEAT / c / f"{r}.npy" for c in ("oceanship", "deepship", "shipsear")
         for r in ("logmel", "spec")]
missing = [str(p.relative_to(FEAT)) for p in need if not p.exists()]
if missing:
    print(f"  MISSING: {missing}")
    raise SystemExit("feature dataset incomplete")

for c in ("oceanship", "deepship", "shipsear"):
    n = sum(1 for _ in open(FEAT / c / "index.csv", encoding="utf-8")) - 1
    gb = sum((FEAT / c / f"{r}.npy").stat().st_size for r in ("logmel", "spec")) / 1e9
    print(f"  {c:<10} {n:>7,} clips   {gb:>6.2f} GB of arrays")
print(f"  splits     : {sorted(p.name for p in (FEAT / 'splits').glob('*'))[:8]}")

mf = FEAT / "manifest.json"
if mf.exists():
    m = json.loads(mf.read_text(encoding="utf-8"))
    print(f"  manifest   : { {k: v.get('rows') for k, v in m.items() if isinstance(v, dict)} }")

SCRIPTS.mkdir(parents=True, exist_ok=True)
RUNS.mkdir(parents=True, exist_ok=True)
try:
    import torch
    gpu = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu only"
    print(f"  torch      : {torch.__version__}  cuda={torch.cuda.is_available()}  {gpu}")
except ImportError:
    # W3-A3 is CPU-only and must still run where torch is absent
    print("  torch      : not installed (fine for the CPU-only notebooks)")
print(f"  free space : {shutil.disk_usage(WORK).free / 1e9:.1f} GB")
tick("setup done")


In [ ]:
# --------------------------------------------------------------------------- #
# find every run directory this session can see
# --------------------------------------------------------------------------- #
import csv as _csv
from pathlib import Path

def find_runs():
    dirs = []
    for base in (WORK, INPUT):
        if not base.is_dir():
            continue
        for depth in range(4):
            for p in sorted(base.glob("/".join(["*"] * depth + ["results.csv"]))):
                # scratch (leading underscore below the search root) is never a
                # run: this is where the --smoke probe lives, and its row carries
                # the same run id as the real configuration
                try:
                    rel = p.relative_to(base)
                except ValueError:
                    rel = p
                if any(str(part).startswith("_") for part in rel.parts):
                    print(f"  [skip] scratch {p}")
                    continue
                if p.parent not in dirs:
                    dirs.append(p.parent)
    return dirs

RUN_DIRS = find_runs()
if not RUN_DIRS:
    raise SystemExit(
        "no results.csv found anywhere. Run W3-A first, or mount its output as a "
        "dataset (Output tab -> Create Dataset) so it lands under /kaggle/input."
    )
print("run directories:")
tot = 0
for d in RUN_DIRS:
    n = sum(1 for _ in open(d / "results.csv", encoding="utf-8")) - 1
    tot += n
    print(f"  {d}   {n} runs")
print(f"  total {tot} run rows")
tick("inputs found")

In [ ]:
%%writefile /kaggle/working/scripts/w3_agg.py
#!/usr/bin/env python3
"""W3 aggregation — turns run rows into the paper's tables and figures.

Three things this file exists to get right.

1. Recording-level bootstrap confidence intervals.
   Test clips are not independent.  One recording contributes dozens of clips
   that share a channel, a background noise field and an operating condition, so
   resampling clips would treat fifty clips from one ship as fifty independent
   pieces of evidence and report an interval that is far too narrow.  We
   resample RECORDINGS with replacement instead.  This is the interval a reviewer
   should ask for, and it is precisely what a recent DeepShip benchmark listed as
   future work.

2. Never an accuracy without its protocol.
   Every row carries the protocol and the measured leak rate, because a number
   quoted without them is meaningless — the same model moves by tens of points
   between protocols.

3. Where the accuracy comes from.
   For the leaky protocols the test set is split into clips whose vessel (and
   recording) was also seen in training and clips whose was not, and the metric
   is reported on each part.  That decomposition is the mechanism behind the
   headline gap: it shows how much of a 99% number is recognition and how much
   is recall of a ship the model already met.

Usage
-----
    python w3_agg.py --data /kaggle/input/under_water_features \
                     --runs /kaggle/working/w3-runs /kaggle/input/w3a-runs \
                     --out /kaggle/working/w3-tables
"""

from __future__ import annotations

import argparse
import csv
import gzip
import json
import math
from collections import defaultdict
from pathlib import Path

import numpy as np

B_BOOT = 1000
CI = 95.0

PROTOCOL_ORDER = ("clip_random", "rec_random", "vessel_disjoint", "temporal", "group_disjoint")
STRICT = {"vessel_disjoint", "temporal", "group_disjoint"}

LABEL_COL = {"oceanship": "label", "deepship": "class_dir", "shipsear": "class_dir"}
REC_COL = {"oceanship": "rec_key", "deepship": "rec_key", "shipsear": "group"}
VESSEL_COL = {"oceanship": "mmsi", "deepship": "ship_name", "shipsear": "group"}
SPLIT_FILE = {"oceanship": "oceanship_splits.csv.gz", "deepship": "deepship_splits.csv.gz"}
SPLIT_KEY = {"oceanship": ["clip_id"], "deepship": ["rec_key", "clip_idx"]}


# --------------------------------------------------------------------------- #
def open_text(path: Path):
    """Read a possibly-gzipped text file by magic bytes (Kaggle expands `.gz`)."""
    path = Path(path)
    with open(path, "rb") as fh:
        head = fh.read(2)
    if head == b"\x1f\x8b":
        return gzip.open(path, "rt", newline="", encoding="utf-8", errors="replace")
    return open(path, "r", newline="", encoding="utf-8", errors="replace")


def find_all(roots, pattern: str) -> list[Path]:
    """Every file named `pattern` under each root, up to three levels deep.

    Deep enough for `/kaggle/input/<slug>/...`, `/kaggle/input/<owner>/<slug>/...`
    and `/kaggle/working/<dir>/...` alike, so a run directory can be passed in,
    or the notebook output mounted as a dataset, without the caller knowing which.
    """
    out: list[Path] = []
    for r in roots:
        r = Path(r)
        if r.is_file() and r.name == pattern:
            out.append(r)
            continue
        if not r.is_dir():
            continue
        for depth in range(4):
            out += sorted(r.glob("/".join(["*"] * depth + [pattern]) if depth else pattern))
    seen, uniq = set(), []
    for p in out:
        if str(p) not in seen:
            seen.add(str(p))
            uniq.append(p)
    return uniq


def is_scratch(p: Path, base: Path | None = None) -> bool:
    """True if the path is scratch — a component below `base` starts with '_'.

    Only the part of the path BELOW the search root counts.  Inspecting the whole
    absolute path would also reject a perfectly good run directory that happens to
    sit under a folder someone named with a leading underscore.
    """
    try:
        rel = p.relative_to(base) if base is not None else Path(p.name)
    except ValueError:
        rel = Path(p.name)
    return any(str(part).startswith("_") for part in rel.parts)


def discover_run_roots() -> list[Path]:
    """Directories holding a results.csv, found under /kaggle/input and /kaggle/working.

    Scratch directories (leading underscore, relative to the root) are excluded.
    The `--smoke` probe writes a results.csv whose run id is identical to the real
    run of the same configuration, so scanning it would replace a 24-epoch
    headline number with a 2-epoch accuracy of 0.14.
    """
    found = []
    for base in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not base.is_dir():
            continue
        for p in find_all([base], "results.csv"):
            if is_scratch(p, base):
                print(f"  [skip] scratch {p}")
                continue
            d = p.parent
            if d not in found:
                found.append(d)
    return found


def macro_f1(y_true, y_pred, n_classes) -> float:
    f1s = []
    for c in range(n_classes):
        tp = int(((y_pred == c) & (y_true == c)).sum())
        fp = int(((y_pred == c) & (y_true != c)).sum())
        fn = int(((y_pred != c) & (y_true == c)).sum())
        if tp + fp + fn == 0:
            continue
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        f1s.append(2 * p * r / (p + r) if p + r else 0.0)
    return float(np.mean(f1s)) if f1s else 0.0


def bootstrap_over_recordings(y_true, y_pred, group, n_classes, b=B_BOOT, seed=0):
    """Resample clusters (recordings) with replacement; return metric CIs."""
    rng = np.random.default_rng(seed)
    uniq, inv = np.unique(group, return_inverse=True)
    if len(uniq) < 2:
        return None
    order = np.argsort(inv, kind="stable")
    bounds = np.searchsorted(inv[order], np.arange(len(uniq) + 1))
    groups = [order[bounds[i]:bounds[i + 1]] for i in range(len(uniq))]
    accs = np.empty(b)
    f1s = np.empty(b)
    for i in range(b):
        pick = rng.integers(0, len(groups), len(groups))
        idx = np.concatenate([groups[j] for j in pick])
        accs[i] = (y_pred[idx] == y_true[idx]).mean()
        f1s[i] = macro_f1(y_true[idx], y_pred[idx], n_classes)
    lo, hi = (100 - CI) / 2, 100 - (100 - CI) / 2
    return {"acc_lo": float(np.percentile(accs, lo)), "acc_hi": float(np.percentile(accs, hi)),
            "f1_lo": float(np.percentile(f1s, lo)), "f1_hi": float(np.percentile(f1s, hi)),
            "n_clusters": int(len(uniq))}


def fmt(x, nd=4):
    return "-" if x is None or (isinstance(x, float) and math.isnan(x)) else f"{x:.{nd}f}"


# --------------------------------------------------------------------------- #
def load_train_identities(root: Path) -> dict:
    """{(corpus, protocol, seed): (train_recordings, train_vessels)} for the leak split.

    Recomputed from the frozen split file and the index, so the decomposition is
    derived from the protocol rather than from metadata smuggled through the
    prediction files.
    """
    out = {}
    for corpus, fname in SPLIT_FILE.items():
        idx_path = root / corpus / "index.csv"
        if not idx_path.exists():
            continue
        with idx_path.open(encoding="utf-8", newline="") as fh:
            idx = list(csv.DictReader(fh))
        keys = SPLIT_KEY[corpus]
        by_key = {tuple(r[k] for k in keys): r for r in idx}

        path = root / "splits" / fname
        if not path.exists():
            alt = root / "splits" / fname[:-3]
            path = alt if alt.exists() else path
        if not path.exists():
            continue
        acc: dict[tuple, dict] = defaultdict(lambda: {"rec": set(), "ves": set()})
        with open_text(path) as fh:
            for r in csv.DictReader(fh):
                if r["split"] != "train":
                    continue
                row = by_key.get(tuple(r[k] for k in keys))
                if row is None:
                    continue
                d = acc[(r["protocol"], int(r["seed"]))]
                d["rec"].add(row.get(REC_COL[corpus], ""))
                d["ves"].add(row.get(VESSEL_COL[corpus], ""))
        for k, v in acc.items():
            out[(corpus, *k)] = (v["rec"], v["ves"])
    return out


def leak_scoped_metrics(npz, train_ids):
    """Accuracy on the parts of the test set that did / did not share an identity."""
    y, p = npz["y_true"], npz["y_pred"]
    trec, tves = train_ids
    ves = np.array([str(x) for x in npz["vessel"]])
    rec = np.array([str(x) for x in npz["rec_key"]])
    out = {"all": (float((p == y).mean()) if len(y) else 0.0, int(len(y)))}
    for name, mask in (("seen_vessel", np.isin(ves, list(tves))),
                       ("unseen_vessel", ~np.isin(ves, list(tves))),
                       ("seen_recording", np.isin(rec, list(trec))),
                       ("unseen_recording", ~np.isin(rec, list(trec)))):
        out[name] = (float((p[mask] == y[mask]).mean()) if mask.sum() else None, int(mask.sum()))
    return out


# --------------------------------------------------------------------------- #
def main() -> int:
    ap = argparse.ArgumentParser(description="Aggregate W3 runs into tables and figures.")
    ap.add_argument("--data", type=Path, required=True, help="features root (for index + splits)")
    ap.add_argument("--runs", type=Path, nargs="*", default=None,
                    help="directories holding results.csv / predictions / embeddings")
    ap.add_argument("--out", type=Path, default=Path("/kaggle/working/w3-tables"))
    ap.add_argument("--b", type=int, default=B_BOOT)
    ap.add_argument("--no-figures", action="store_true")
    args = ap.parse_args()

    roots = args.runs or discover_run_roots()
    # the same scratch rule applies to an explicitly passed directory, so
    # `--runs /kaggle/working/w3-runs /kaggle/working/_probe` cannot smuggle a
    # 2-epoch probe row into the headline table
    dropped = [r for r in roots if str(Path(r).name).startswith("_")]
    for d in dropped:
        print(f"  [skip] scratch {d}")
    roots = [r for r in roots if r not in dropped and Path(r).exists()]
    if not roots:
        raise SystemExit(
            "no run directory found. Pass --runs, or mount a previous notebook's "
            "output as a dataset so it lands under /kaggle/input."
        )
    args.out.mkdir(parents=True, exist_ok=True)
    (args.out / "tables").mkdir(exist_ok=True)
    (args.out / "figures").mkdir(exist_ok=True)

    print("=" * 100)
    print("W3 aggregation")
    print("=" * 100)
    print(f"  features : {args.data}")
    print(f"  run dirs : {[str(r) for r in roots]}")

    # ------------------------------------------------------------------ results
    rows: list[dict] = []
    pred_dirs: list[Path] = []
    for r in roots:
        for res in find_all([Path(r)], "results.csv"):
            with res.open(encoding="utf-8", newline="") as fh:
                got = list(csv.DictReader(fh))
            print(f"  {res}  -> {len(got)} rows")
            rows += got
        for pd in sorted(Path(r).glob("*/predictions")) + sorted(Path(r).glob("predictions")):
            pred_dirs.append(pd)
    if not rows:
        raise SystemExit("no results.csv rows found")

    # Same run id in two places: keep the better-trained one and say so.  This is
    # the guard for a probe or a partial run having been picked up from a mounted
    # dataset, and it is deliberately not silent.
    by_id: dict[str, dict] = {}
    clashes = []
    for r in rows:
        old = by_id.get(r["run_id"])
        if old is None:
            by_id[r["run_id"]] = r
            continue
        try:
            new_key = (int(r.get("epochs") or 0), int(r.get("n_train") or 0))
            old_key = (int(old.get("epochs") or 0), int(old.get("n_train") or 0))
        except (TypeError, ValueError):
            new_key, old_key = (0, 0), (1, 0)
        if new_key > old_key:
            by_id[r["run_id"]] = r
        clashes.append((r["run_id"], old_key, new_key))
    if clashes:
        print(f"  [warn] {len(clashes)} duplicated run id(s); keeping the longest-trained row")
        for rid, a, b in clashes[:5]:
            print(f"          {rid}  kept {(max(a, b),)}  discarded {(min(a, b),)}")
    rows = list(by_id.values())
    print(f"  {len(rows)} unique runs")

    train_ids = load_train_identities(args.data)
    print(f"  train identity sets for {len(train_ids)} (corpus, protocol, seed) combos")

    # ---------------------------------------------------------- bootstrap CIs
    ci_rows, scope_rows = [], []
    for r in rows:
        run_id = r["run_id"]
        npz_path = None
        for pd in pred_dirs:
            cand = pd / f"{run_id}.npz"
            if cand.exists():
                npz_path = cand
                break
        if npz_path is None:
            continue
        d = np.load(npz_path, allow_pickle=True)
        n_classes = int(r["n_classes"])
        ci = bootstrap_over_recordings(d["y_true"], d["y_pred"], d["rec_key"],
                                       n_classes, b=args.b, seed=int(r["seed"]))
        if ci:
            ci_rows.append({"run_id": run_id, **ci})
        ids = train_ids.get((r["corpus"], r["protocol"], int(r["seed"])))
        if ids is not None:
            m = leak_scoped_metrics(d, ids)
            scope_rows.append({"run_id": run_id, "protocol": r["protocol"],
                               "corpus": r["corpus"], "rep": r["rep"],
                               "backbone": r["backbone"], "seed": r["seed"],
                               **{f"acc_{k}": v[0] for k, v in m.items()},
                               **{f"n_{k}": v[1] for k, v in m.items()}})

    ci_by_run = {c["run_id"]: c for c in ci_rows}
    if ci_rows:
        with (args.out / "tables" / "bootstrap_ci.csv").open("w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=list(ci_rows[0].keys()))
            w.writeheader()
            w.writerows(ci_rows)
    if scope_rows:
        with (args.out / "tables" / "leak_scope.csv").open("w", newline="", encoding="utf-8") as fh:
            w = csv.DictWriter(fh, fieldnames=list(scope_rows[0].keys()))
            w.writeheader()
            w.writerows(scope_rows)

    # ------------------------------------------------------------------ export
    flat = args.out / "tables" / "all_runs.csv"
    cols = sorted({k for r in rows for k in r})
    with flat.open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=cols)
        w.writeheader()
        w.writerows(rows)

    L: list[str] = []
    add = L.append
    add("=" * 100)
    add("W3 tables")
    add("=" * 100)
    add("")
    add(f"runs: {len(rows)}   bootstrap: {args.b} resamples over RECORDINGS, {CI:.0f}% CI")
    add("")

    groups: dict[tuple, list[dict]] = defaultdict(list)
    for r in rows:
        groups[(r["corpus"], r["protocol"], r["rep"], r["backbone"], r["method"],
                r["domain"], r["shuffled"])].append(r)

    def agg(rs: list[dict], metric="test_acc"):
        v = np.array([float(x[metric]) for x in rs if x.get(metric) not in (None, "", "None")])
        return (v.mean(), v.std(ddof=0), len(v)) if len(v) else (float("nan"), float("nan"), 0)

    # -------------------------------------------------- T1 the headline: protocol
    add("=" * 100)
    add("T2  Headline — the same model, different split units")
    add("=" * 100)
    add("")
    add(f"  {'corpus':<10}{'rep':<7}{'backbone':<16}{'method':<8}{'protocol':<17}"
        f"{'acc mean':>9}{'sd':>8}{'n':>4}{'F1':>8}{'95% CI (recording bootstrap)':>32}{'leak%':>8}")
    for (corpus, proto, rep, backbone, method, dom, shuf), rs in sorted(groups.items()):
        if shuf in ("1", 1) or method != "erm" or dom != "none":
            continue
        a, sd, n = agg(rs)
        f1 = agg(rs, "test_macro_f1")[0]
        cis = [ci_by_run[x["run_id"]] for x in rs if x["run_id"] in ci_by_run]
        ci_s = "-"
        if cis:
            lo = np.mean([c["acc_lo"] for c in cis])
            hi = np.mean([c["acc_hi"] for c in cis])
            ci_s = f"[{lo:.4f}, {hi:.4f}]"
        leak = np.mean([float(x["leak_rec_pct"]) for x in rs]) if rs[0].get("leak_rec_pct") else None
        add(f"  {corpus:<10}{rep:<7}{backbone:<16}{method:<6}{proto:<17}"
            f"{fmt(a):>9}{fmt(sd):>8}{n:>4}{fmt(f1):>8}{ci_s:>32}"
            f"{fmt(leak, 1) if leak is not None else '-':>8}")
    add("")

    # --------------------------------------------------- T2 the inflation gap
    add("=" * 100)
    add("T3  The inflation gap — accuracy lost when the split unit is made honest")
    add("=" * 100)
    add("")
    add(f"  {'corpus':<10}{'rep':<7}{'backbone':<16}"
        f"{'clip_random':>13}{'rec_random':>13}{'strict':>11}{'P1-P2':>9}{'P1-strict':>11}")
    for corpus in sorted({r["corpus"] for r in rows}):
        for rep in sorted({r["rep"] for r in rows if r["corpus"] == corpus}):
            for backbone in sorted({r["backbone"] for r in rows
                                    if r["corpus"] == corpus and r["rep"] == rep}):
                def mean_of(proto):
                    rs = [x for x in rows if x["corpus"] == corpus and x["rep"] == rep
                          and x["backbone"] == backbone and x["protocol"] == proto
                          and x["method"] == "erm" and x["shuffled"] in ("0", 0)]
                    return agg(rs)[0] if rs else None
                a1 = mean_of("clip_random")
                a2 = mean_of("rec_random")
                strict = [mean_of(p) for p in ("vessel_disjoint", "temporal", "group_disjoint")]
                a3 = next((s for s in strict if s is not None), None)
                if a1 is None and a3 is None:
                    continue
                add(f"  {corpus:<10}{rep:<7}{backbone:<16}{fmt(a1):>13}{fmt(a2):>13}{fmt(a3):>11}"
                    f"{fmt(None if (a1 is None or a2 is None) else a1 - a2):>9}"
                    f"{fmt(None if (a1 is None or a3 is None) else a1 - a3):>11}")
    add("")

    # ------------------------------- T3 where the accuracy comes from (mechanism)
    if scope_rows:
        add("=" * 100)
        add("T4  Where the accuracy comes from — test clips whose ship was already seen")
        add("=" * 100)
        add("")
        add("  For a leaky protocol the test set is not one population: some clips come")
        add("  from a ship the model trained on, some do not. Reporting one number hides")
        add("  the fact that the two parts behave completely differently.")
        add("")
        add(f"  {'corpus':<10}{'protocol':<17}{'all':>8}{'seen vessel':>13}{'unseen vessel':>15}"
            f"{'seen rec':>10}{'unseen rec':>12}")
        ag: dict[tuple, list[dict]] = defaultdict(list)
        for x in scope_rows:
            ag[(x["corpus"], x["protocol"])].append(x)
        for (corpus, proto), rs in sorted(ag.items()):
            def m(k):
                v = [x[f"acc_{k}"] for x in rs
                     if x.get(f"acc_{k}") not in (None, "", "None")]
                return float(np.mean(v)) if v else None
            add(f"  {corpus:<10}{proto:<17}{fmt(m('all')):>8}{fmt(m('seen_vessel')):>13}"
                f"{fmt(m('unseen_vessel')):>15}{fmt(m('seen_recording')):>10}"
                f"{fmt(m('unseen_recording')):>12}")
        add("")

    # ------------------------------------------------------------- T4 per-seed
    add("=" * 100)
    add("T5  Per-seed spread (the test set size is a random variable — never quote one seed)")
    add("=" * 100)
    add("")
    add(f"  {'corpus':<10}{'protocol':<30}{'acc per seed':<46}{'n_test per seed':<28}")
    per = defaultdict(list)
    for r in rows:
        if (r["method"] == "erm" and r["domain"] == "none" and r["shuffled"] in ("0", 0)
                and r["backbone"] == "cnn_small" and r["rep"] == "logmel"):
            per[(r["corpus"], r["protocol"])].append(r)
    for (corpus, proto), rs in sorted(per.items()):
        rs = sorted(rs, key=lambda x: int(x["seed"]))
        accs = [float(x["test_acc"]) for x in rs]
        ns = [int(x["n_test"]) for x in rs]
        add(f"  {corpus:<10}{proto:<30}{str([round(a, 3) for a in accs]):<46}{str(ns):<28}")
        if ns:
            add(f"  {'':<40}mean {np.mean(accs):.4f}  sd {np.std(accs):.4f}"
                f"   n_test {min(ns)}-{max(ns)} (mean {np.mean(ns):.0f}, "
                f"spread {100*(max(ns)-min(ns))/max(np.mean(ns),1):.0f}%)")
    add("")

    # ------------------------------------------------------------ T5 scale
    add("=" * 100)
    add("T6  Parameter count vs strict-protocol accuracy")
    add("=" * 100)
    add("")
    add("  Reproduces or contradicts Soti et al. (2026): no gain from 11.17M over ~157K.")
    add("")
    add(f"  {'corpus':<10}{'backbone':<16}{'params':>12}{'clip_random':>13}{'rec_random':>13}"
        f"{'strict':>11}")
    for corpus in sorted({r["corpus"] for r in rows}):
        for backbone in sorted({r["backbone"] for r in rows if r["corpus"] == corpus}):
            rs = [x for x in rows if x["corpus"] == corpus and x["backbone"] == backbone
                  and x["method"] == "erm" and x["domain"] == "none"
                  and x["shuffled"] in ("0", 0) and x["rep"] == "logmel"]
            if not rs:
                continue
            params = max(int(x["n_params"]) for x in rs)

            def mm(proto):
                s = [x for x in rs if x["protocol"] == proto]
                return agg(s)[0] if s else None
            strict = [mm(p) for p in ("vessel_disjoint", "temporal", "group_disjoint")]
            add(f"  {corpus:<10}{backbone:<16}{params:>12,}{fmt(mm('clip_random')):>13}"
                f"{fmt(mm('rec_random')):>13}"
                f"{fmt(next((v for v in strict if v is not None), None)):>11}")
    add("")

    # -------------------------------------------------------------- T6 methods
    dg = [r for r in rows if r["method"] != "erm" or r["domain"] != "none"]
    if dg:
        add("=" * 100)
        add("T7  Domain generalisation (C4) — domain labels from metadata, target domain unseen")
        add("=" * 100)
        add("")
        add("  `vessel_hash` is the NEGATIVE CONTROL: real partitions, no acoustic meaning.")
        add("  A gain there means the regulariser, not the domain structure, did the work.")
        add("")
        add(f"  {'corpus':<10}{'protocol':<17}{'rep':<7}{'backbone':<12}{'method':<7}"
            f"{'domain':<14}{'acc mean':>9}{'sd':>8}{'n':>4}{'vs ERM':>9}")
        base = {}
        for r in rows:
            if r["method"] == "erm" and r["domain"] == "none" and r["shuffled"] in ("0", 0):
                base[(r["corpus"], r["protocol"], r["rep"], r["backbone"])] = r
        g2: dict[tuple, list[dict]] = defaultdict(list)
        for r in dg:
            g2[(r["corpus"], r["protocol"], r["rep"], r["backbone"], r["method"],
                r["domain"])].append(r)
        for k, rs in sorted(g2.items()):
            a, sd, n = agg(rs)
            b = base.get(k[:4])
            delta = (a - float(b["test_acc"])) if b else None
            add(f"  {k[0]:<10}{k[1]:<17}{k[2]:<7}{k[3]:<12}{k[4]:<7}{k[5]:<14}"
                f"{fmt(a):>9}{fmt(sd):>8}{n:>4}{fmt(delta):>9}")
        add("")

    # ------------------------------------------------------ T7 cross-corpus
    cross = [r for r in rows if r["protocol"].startswith("cross_")]
    if cross:
        add("=" * 100)
        add("T8  Cross-corpus (unseen corpus) — shared 4-class vocabulary")
        add("=" * 100)
        add("")
        add(f"  {'direction':<34}{'rep':<7}{'backbone':<16}{'acc':>8}{'macro F1':>10}"
            f"{'chance':>8}{'n_test':>9}{'vocab_approx':>14}")
        for r in sorted(cross, key=lambda x: x["run_id"]):
            add(f"  {r['protocol']:<34}{r['rep']:<7}{r['backbone']:<16}"
                f"{fmt(float(r['test_acc'])):>8}{fmt(float(r['test_macro_f1'])):>10}"
                f"{fmt(float(r['chance_acc'])):>8}{r['n_test']:>9}{r.get('vocab_approx', '0'):>14}")
        add("")

    # --------------------------------------------------------- T8 diagnostics
    diag = [r for r in rows if r["shuffled"] in ("1", 1)]
    if diag:
        add("=" * 100)
        add("T9  Label-shuffling control (required by Hummel et al. 2026)")
        add("=" * 100)
        add("")
        add("  Training labels permuted, everything else identical.  A model that scores")
        add("  above chance here is not learning the task — it is reading a leak or a")
        add("  shortcut out of the split or the padding.")
        add("")
        add(f"  {'corpus':<10}{'protocol':<17}{'rep':<7}{'backbone':<16}{'acc':>8}{'chance':>8}"
            f"{'n_test':>9}")
        for r in sorted(diag, key=lambda x: x["run_id"]):
            add(f"  {r['corpus']:<10}{r['protocol']:<17}{r['rep']:<7}{r['backbone']:<16}"
                f"{fmt(float(r['test_acc'])):>8}{fmt(float(r['chance_acc'])):>8}{r['n_test']:>9}")
        add("")

    # ------------------------------------------------------------- T1 leakage
    for corpus in CORPORA_OK:
        p = args.data / "splits" / f"{corpus}_leakage_summary.csv"
        if p.exists():
            add("=" * 100)
            add("T1  Leakage audit (frozen protocol) — the table the paper opens with")
            add("=" * 100)
            add("")
            with open_text(p) as fh:
                rdr = list(csv.DictReader(fh))
            add(f"  {'protocol':<18}{'seed':>5}{'n_test':>9}{'n_ves':>7}{'n_rec':>7}"
                f"{'share vessel':>14}{'share rec':>11}{'share clip':>12}")
            for r in rdr:
                add(f"  {r['protocol']:<18}{r['seed']:>5}{r['n_test']:>9}"
                    f"{r['n_test_vessels']:>7}{r['n_test_recordings']:>7}"
                    f"{r['pct_test_share_vessel'] + '%':>14}"
                    f"{r['pct_test_share_recording'] + '%':>11}"
                    f"{r['pct_test_share_clip'] + '%':>12}")
            add("")

    text = "\n".join(L)
    print()
    print(text)
    (args.out / "tables" / "w3_tables.txt").write_text(text, encoding="utf-8")

    # ------------------------------------------------------------------ figures
    if not args.no_figures:
        try:
            make_figures(rows, groups, ci_by_run, scope_rows, args.out / "figures")
        except Exception as exc:  # a missing font must not lose the tables
            print(f"[warn] figures failed: {type(exc).__name__}: {exc}")

    print(f"\nwritten: {args.out}")
    return 0


CORPORA_OK = ("oceanship", "deepship", "shipsear")


def make_figures(rows, groups, ci_by_run, scope_rows, outdir: Path):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    def agg(rs, metric="test_acc"):
        v = np.array([float(x[metric]) for x in rs if x.get(metric) not in (None, "", "None")])
        return (v.mean(), v.std(ddof=0)) if len(v) else (float("nan"), 0.0)

    # F1: the protocol gap per backbone
    sel = [k for k in groups if k[4] == "erm" and k[6] in ("0", 0) and k[2] == "logmel"]
    if sel:
        protos = [p for p in PROTOCOL_ORDER]
        fig, ax = plt.subplots(figsize=(9, 5))
        labels, vals, errs = [], [], []
        for k in sorted(sel):
            corpus, proto = k[0], k[1]
            if proto not in protos:
                continue
            a, sd = agg(groups[k])
            labels.append(f"{corpus}\n{k[3]}\n{proto}")
            vals.append(a)
            errs.append(sd)
        if labels:
            ax.bar(range(len(vals)), vals, yerr=errs, capsize=3)
            ax.set_xticks(range(len(labels)))
            ax.set_xticklabels(labels, fontsize=7, rotation=90)
            ax.set_ylabel("test accuracy")
            ax.set_ylim(0, 1)
            ax.set_title("Same model, same recipe — only the split unit changes")
            fig.tight_layout()
            fig.savefig(outdir / "F2_protocol_gap.png", dpi=160)
            plt.close(fig)

    # F2: where the accuracy comes from
    if scope_rows:
        ag = defaultdict(list)
        for x in scope_rows:
            ag[x["protocol"]].append(x)
        keys = ["all", "seen_vessel", "unseen_vessel"]
        fig, ax = plt.subplots(figsize=(8, 4.5))
        protos = [p for p in PROTOCOL_ORDER if p in ag]
        w = 0.25
        for i, kk in enumerate(keys):
            vals = []
            for p in protos:
                v = [x[f"acc_{kk}"] for x in ag[p] if x.get(f"acc_{kk}") not in (None, "", "None")]
                vals.append(float(np.mean(v)) if v else 0.0)
            ax.bar(np.arange(len(protos)) + (i - 1) * w, vals, width=w, label=kk)
        ax.set_xticks(range(len(protos)))
        ax.set_xticklabels(protos)
        ax.set_ylabel("test accuracy")
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
        ax.set_title("Accuracy on test clips whose ship WAS vs WAS NOT seen in training")
        fig.tight_layout()
        fig.savefig(outdir / "F3_leak_scope.png", dpi=160)
        plt.close(fig)

    # F3: scale curve
    fig, ax = plt.subplots(figsize=(7, 5))
    plotted = False
    for corpus in sorted({r["corpus"] for r in rows}):
        for proto in PROTOCOL_ORDER:
            pts = {}
            for r in rows:
                if (r["corpus"] == corpus and r["protocol"] == proto and r["rep"] == "logmel"
                        and r["method"] == "erm" and r["shuffled"] in ("0", 0)):
                    pts.setdefault(int(r["n_params"]), []).append(float(r["test_acc"]))
            if len(pts) < 2:
                continue
            xs = sorted(pts)
            ys = [float(np.mean(pts[x])) for x in xs]
            ax.semilogx(xs, ys, marker="o", label=f"{corpus} / {proto}")
            plotted = True
    if plotted:
        ax.set_xlabel("trainable parameters")
        ax.set_ylabel("test accuracy")
        ax.set_ylim(0, 1)
        ax.legend(fontsize=8)
        ax.set_title("Does a bigger model help once the split is honest?")
        fig.tight_layout()
        fig.savefig(outdir / "F4_scale.png", dpi=160)
    plt.close(fig)


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /kaggle/working/scripts/w3_diag.py
#!/usr/bin/env python3
"""W3 diagnostics — the checks a reviewer is entitled to ask for.

Everything here runs on frozen embeddings dumped by `w3_run.py --embeddings`, so
none of it needs a GPU and all of it is cheap.

The four questions
------------------
1. How much *recording identity* survives in the representation?  (P8)
   Hummel et al. (2026, Ecological Informatics 95 103709) make this mandatory
   reporting for passive-acoustic deep learning: if a linear read-out of the
   frozen features can name the recording a clip came from, then any downstream
   accuracy is partly session recall rather than ship recognition.  Reported as a
   nearest-centroid balanced accuracy plus normalised mutual information against
   the true recording id — and the same pipeline run on shuffled keys, so the
   reader can see what "no signal" looks like on this data size.

2. How separable are the *domain* labels we intend to be invariant to?  (P6)
   An invariance objective can only help if the domains are distinguishable in
   the features.  If `date` is already invisible, a gain from IRM/VREx on it
   would be unexplained and should be treated as a red flag, not a result.

3. Is *duration* a shortcut?  (P5)
   Short clips are padded rather than dropped, and duration is class-correlated
   (Oceanship Passenger median 3.67 s against Tug 5.00 s).  A model can therefore
   read the class off the amount of padding.  Two proxies are measured: the
   stored source duration, and the fraction of time frames sitting on the
   log-mel floor — the latter being what the network sees in the image with no
   acoustic understanding at all.  Both are scored out-of-fold.

4. Does the label-shuffling control hold?  (P7)
   Summarised from the run rows.

Usage
-----
    python w3_diag.py --data /kaggle/input/under_water_features \
                      --runs /kaggle/working/w3-runs \
                      --out /kaggle/working/w3-tables
"""

from __future__ import annotations

import argparse
import csv
import gzip
import json
import re
from collections import Counter
from pathlib import Path

import numpy as np

LABEL_COL = {"oceanship": "label", "deepship": "class_dir", "shipsear": "class_dir"}
REC_COL = {"oceanship": "rec_key", "deepship": "rec_key", "shipsear": "group"}
DATE_RE = re.compile(r"(\d{4})[-/]?(\d{2})")


# --------------------------------------------------------------------------- #
def open_text(path: Path):
    path = Path(path)
    with open(path, "rb") as fh:
        head = fh.read(2)
    if head == b"\x1f\x8b":
        return gzip.open(path, "rt", newline="", encoding="utf-8", errors="replace")
    return open(path, "r", newline="", encoding="utf-8", errors="replace")


def entropy(counts: np.ndarray) -> float:
    p = counts[counts > 0].astype(np.float64)
    p = p / p.sum()
    return float(-(p * np.log(p)).sum())


def mutual_information(a: np.ndarray, b: np.ndarray) -> float:
    """I(A;B) in nats, straight from the contingency table."""
    ua, ia = np.unique(a, return_inverse=True)
    ub, ib = np.unique(b, return_inverse=True)
    table = np.zeros((len(ua), len(ub)), dtype=np.int64)
    np.add.at(table, (ia, ib), 1)
    n = table.sum()
    if n == 0:
        return 0.0
    pij = table / n
    pi, pj = pij.sum(1, keepdims=True), pij.sum(0, keepdims=True)
    with np.errstate(divide="ignore", invalid="ignore"):
        terms = pij * (np.log(pij) - np.log(pi) - np.log(pj))
    return float(np.nansum(terms))


def nmi(a: np.ndarray, b: np.ndarray) -> float:
    """Normalised mutual information in [0, 1] (geometric-mean normalisation)."""
    i = mutual_information(a, b)
    ha = entropy(np.bincount(np.unique(a, return_inverse=True)[1]))
    hb = entropy(np.bincount(np.unique(b, return_inverse=True)[1]))
    if ha <= 0 or hb <= 0:
        return 0.0
    return float(i / np.sqrt(ha * hb))


def balanced_accuracy(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    r = [float((y_pred[y_true == c] == c).mean()) for c in np.unique(y_true)]
    return float(np.mean(r)) if r else 0.0


def centroid_predict(Xt, yt, Xq, min_per_class=2, max_classes=0):
    """Nearest class-centroid read-out on L2-normalised features.

    Centroid matching rather than multinomial logistic regression on purpose: the
    recording probe has thousands of classes, where a multinomial fit would be
    both slow and numerically unhappy.  Cosine-to-centroid is a linear read-out
    of the same features and is what makes this tractable at Oceanship's scale.
    """
    def norm(a):
        return a / np.maximum(np.linalg.norm(a, axis=1, keepdims=True), 1e-8)

    yt = np.asarray(yt)
    Xt, Xq = norm(np.asarray(Xt, dtype=np.float32)), norm(np.asarray(Xq, dtype=np.float32))
    counts = Counter(yt.tolist())
    keep = [c for c, n in counts.items() if n >= min_per_class]
    if max_classes and len(keep) > max_classes:
        keep = sorted(keep, key=lambda c: -counts[c])[:max_classes]
    if len(keep) < 2:
        return None, None
    keep_arr = np.array(keep)
    cen = np.stack([Xt[yt == c].mean(0) for c in keep])
    cen = norm(cen)
    pred = keep_arr[np.argmax(Xq @ cen.T, axis=1)]
    return pred, keep_arr


def out_of_fold_1d(x: np.ndarray, y: np.ndarray, folds: int = 5) -> float:
    """Balanced accuracy of a nearest-class-mean rule in ONE feature, out of fold.

    Resubstitution would be optimistic here, and this probe exists precisely to
    bound how much a shortcut is worth, so it has to be scored honestly.
    """
    rng = np.random.default_rng(0)
    order = rng.permutation(len(x))
    parts = np.array_split(order, folds)
    pred = np.empty(len(x), dtype=object)
    for i in range(folds):
        te = parts[i]
        tr = np.concatenate([parts[j] for j in range(folds) if j != i])
        means = {c: float(x[tr][y[tr] == c].mean()) for c in np.unique(y[tr])}
        cs = sorted(means)
        mu = np.array([means[c] for c in cs])
        pred[te] = [cs[j] for j in np.argmin(np.abs(x[te][:, None] - mu[None, :]), axis=1)]
    return balanced_accuracy(y, pred.astype(y.dtype))


def duration_bucket(x: np.ndarray, step: float = 0.1) -> np.ndarray:
    return np.round(np.asarray(x, dtype=np.float64) / step).astype(np.int64)


def date_key_of(rec_key: str) -> str:
    m = DATE_RE.search(str(rec_key) or "")
    return f"{m.group(1)}-{m.group(2)}" if m else "unknown"


def top_buckets(arr: np.ndarray, k: int = 4) -> np.ndarray:
    top = [c for c, _ in Counter(arr.tolist()).most_common(k)]
    return np.where(np.isin(arr, top), arr, "other")


# --------------------------------------------------------------------------- #
def main() -> int:
    ap = argparse.ArgumentParser(description="W3 diagnostics on frozen embeddings.")
    ap.add_argument("--data", type=Path, required=True)
    ap.add_argument("--runs", type=Path, nargs="*", default=None)
    ap.add_argument("--out", type=Path, default=Path("/kaggle/working/w3-tables"))
    ap.add_argument("--max-runs", type=int, default=6)
    ap.add_argument("--max-classes", type=int, default=3000,
                    help="cap the recording probe's class count (bounds runtime)")
    args = ap.parse_args()

    files: list[Path] = []
    for root in [Path(r) for r in (args.runs or ["/kaggle/working/w3-runs"])]:
        for depth in range(4):
            files += sorted(root.glob("/".join(["*"] * depth + ["embeddings", "*.npz"])))
    uniq, seen = [], set()
    for p in files:
        if p.name not in seen:
            seen.add(p.name)
            uniq.append(p)
    if not uniq:
        raise SystemExit(
            "no embedding .npz found. Run w3_run.py with --embeddings, or point "
            "--runs at a directory containing an embeddings/ folder."
        )
    uniq = uniq[: args.max_runs]

    (args.out / "tables").mkdir(parents=True, exist_ok=True)
    L: list[str] = []
    add = L.append
    add("=" * 100)
    add("W3 diagnostics")
    add("=" * 100)
    add(f"  {len(uniq)} embedding files from {args.runs or ['/kaggle/working/w3-runs']}")
    summary: dict = {}

    for f in uniq:
        d = np.load(f, allow_pickle=True)
        corpus = str(d["corpus"]) if "corpus" in d else "oceanship"
        add("")
        add("=" * 100)
        add(f"run: {corpus} / {d['rep']} / {d['backbone']} / {d['protocol']} / s{d['seed']}")
        add("=" * 100)
        tr_e = d["train_emb"].astype(np.float32)
        te_e = d["test_emb"].astype(np.float32)
        te_y = np.asarray(d["test_y"])
        chance_cls = 1.0 / max(len(set(te_y.tolist())), 1)
        add(f"  embeddings: train {tr_e.shape}  test {te_e.shape}   "
            f"test classes {len(set(te_y.tolist()))} (chance {chance_cls:.4f})")
        rng = np.random.default_rng(0)

        # ------------------------------------------------ the class read-out
        pred, _ = centroid_predict(tr_e, d["train_y"], te_e)
        if pred is not None:
            add(f"  class read-out        balanced acc = {balanced_accuracy(te_y, pred):.4f}"
                f"   NMI = {nmi(te_y, pred):.4f}")

        # --------------- P8 recording / vessel identity in the representation
        add("")
        add("  P8  identity surviving in the frozen representation "
            "(mandatory reporting, Hummel et al. 2026)")
        for name, tk, qk in (("recording", "rec", "rec"), ("vessel", "vessel", "vessel")):
            tr_key = np.asarray([str(x) for x in d[f"train_{tk}"]])
            te_key = np.asarray([str(x) for x in d[f"test_{tk}"]])
            pred, keep = centroid_predict(tr_e, tr_key, te_e, max_classes=args.max_classes)
            if pred is None:
                add(f"      {name:<10} too few classes with >=2 training clips")
                continue
            m = np.isin(te_key, keep)
            ba, mi = balanced_accuracy(te_key[m], pred[m]), nmi(te_key[m], pred[m])
            pred_s, keep_s = centroid_predict(tr_e, rng.permutation(tr_key), te_e,
                                              max_classes=args.max_classes)
            ms = np.isin(te_key, keep_s)
            ba_s = balanced_accuracy(te_key[ms], pred_s[ms])
            add(f"      {name:<10} classes={len(keep):>6,}  covered test clips={int(m.sum()):>7,}"
                f"   balanced acc = {ba:.4f}   NMI = {mi:.4f}")
            add(f"      {'':<10} shuffled-key control: balanced acc = {ba_s:.4f}"
                f"   (the floor this data size can reach by chance)")
            summary[f"{f.stem}|{name}_balanced_acc"] = round(ba, 4)
            summary[f"{f.stem}|{name}_nmi"] = round(mi, 4)
            summary[f"{f.stem}|{name}_shuffled_balanced_acc"] = round(ba_s, 4)

        # -------------------------------------------------- P6 domain labels
        add("")
        add("  P6  are the domain labels we want invariance to actually visible?")
        candidates = []
        if "train_official" in d:
            t = np.asarray([str(x) for x in d["train_official"]])
            q = np.asarray([str(x) for x in d["test_official"]])
            if len(set(t.tolist())) > 1:
                candidates.append(("official split", t, q))
        if "train_date" in d:
            t = np.asarray([str(x) for x in d["train_date"]])
            q = np.asarray([str(x) for x in d["test_date"]])
            if len(set(t.tolist())) > 1:
                candidates.append(("date (index column)", t, q))
        t = np.asarray([date_key_of(x) for x in d["train_rec"]])
        q = np.asarray([date_key_of(x) for x in d["test_rec"]])
        if len(set(t.tolist())) > 1:
            candidates.append(("date (from rec_key)", t, q))
        if not candidates:
            add("      no usable domain label for this corpus")
        for name, tk, qk in candidates:
            pred, keep = centroid_predict(tr_e, tk, te_e, min_per_class=2)
            if pred is None:
                add(f"      {name:<22} too few groups")
                continue
            m = np.isin(qk, keep)
            ba = balanced_accuracy(qk[m], pred[m])
            ch = 1.0 / max(len(set(qk[m].tolist())), 1)
            add(f"      {name:<22} groups={len(keep):<3} balanced acc = {ba:.4f}  "
                f"NMI = {nmi(qk[m], pred[m]):.4f}  (chance {ch:.4f})")
            summary[f"{f.stem}|domain_{name.split()[0]}_balanced_acc"] = round(ba, 4)

    # --------------------------------------------------- P5 duration shortcut
    add("")
    add("=" * 100)
    add("P5  duration / padding as a shortcut")
    add("=" * 100)
    add("")
    add("  Short clips are padded, not dropped, and duration is class-correlated, so a")
    add("  model can read the class off the amount of padding.  Both probes are scored")
    add("  out-of-fold; `chance` is 1/K.")
    add("")
    add(f"  {'corpus':<10}{'proxy':<26}{'K':>4}{'balanced acc':>14}{'NMI':>9}{'chance':>9}")
    for corpus in ("oceanship", "deepship", "shipsear"):
        idx_path = args.data / corpus / "index.csv"
        if not idx_path.exists():
            continue
        with idx_path.open(encoding="utf-8", newline="") as fh:
            rows = list(csv.DictReader(fh))
        y = np.array([r[LABEL_COL[corpus]] for r in rows])
        K = len(set(y.tolist()))
        dur = np.array([float(r.get("src_dur_s") or 0.0) for r in rows])
        add(f"  {corpus:<10}{'src_dur_s alone':<26}{K:>4}"
            f"{out_of_fold_1d(dur, y):>14.4f}{nmi(y, duration_bucket(dur)):>9.4f}{1/K:>9.4f}")
        summary[f"{corpus}|duration_balanced_acc"] = round(out_of_fold_1d(dur, y), 4)

        arr_p = args.data / corpus / "logmel.npy"
        if arr_p.exists():
            arr = np.load(arr_p, mmap_mode="r")
            n = min(4000, len(rows))
            idx = np.linspace(0, len(rows) - 1, n).astype(int)
            frac = np.empty(n, dtype=np.float32)
            for j, k in enumerate(idx):
                col = np.asarray(arr[int(k)], dtype=np.float32).mean(0)
                frac[j] = float((col <= -79.0).mean())
            y_s = y[idx]
            add(f"  {corpus:<10}{'floor-frame fraction':<26}{K:>4}"
                f"{out_of_fold_1d(frac, y_s):>14.4f}"
                f"{nmi(y_s, duration_bucket(frac)):>9.4f}{1/K:>9.4f}")
            summary[f"{corpus}|floorfrac_balanced_acc"] = round(
                out_of_fold_1d(frac, y_s), 4)
            del arr

    # --------------------------------------------------------- P7 summary
    add("")
    add("=" * 100)
    add("P7  label-shuffling control")
    add("=" * 100)
    add("")
    n_shuf = 0
    for f in uniq:
        d = np.load(f, allow_pickle=True)
        if int(d["shuffled"]) if "shuffled" in d else 0:
            n_shuf += 1
            add(f"  {f.stem} is a shuffled control — compare its test accuracy "
                f"against chance in w3_tables.txt (table T9)")
    if not n_shuf:
        add("  no shuffled control run present; run w3_run.py --shuffle-labels")
    add("")
    add("  A shuffled run that sits at chance is the evidence that neither the")
    add("  protocol nor the padding is handing the model the answer.")

    text = "\n".join(L)
    print(text)
    (args.out / "tables" / "w3_diagnostics.txt").write_text(text, encoding="utf-8")
    (args.out / "tables" / "w3_diagnostics.json").write_text(
        json.dumps(summary, indent=2, sort_keys=True), encoding="utf-8")
    print(f"\nwritten: {args.out}/tables")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
# --------------------------------------------------------------------------- #
# the tables and the figures
# --------------------------------------------------------------------------- #
TABLES = WORK / "w3-tables"
cmd = [sys.executable, str(SCRIPTS / "w3_agg.py"), "--data", str(FEAT),
       "--out", str(TABLES), "--runs"] + [str(d) for d in RUN_DIRS]
print("$ " + " ".join(cmd), flush=True)
subprocess.run(cmd, check=True)

In [ ]:
# diagnostics, if any embedding dumps were produced by W3-C
emb = []
for d in RUN_DIRS:
    for depth in range(3):
        emb += sorted(d.glob("/".join(["*"] * depth + ["embeddings"])))
if emb:
    cmd = [sys.executable, str(SCRIPTS / "w3_diag.py"), "--data", str(FEAT),
           "--runs", str(RUN_DIRS[0]), "--out", str(TABLES)]
    for d in RUN_DIRS[1:]:
        cmd += ["--runs", str(d)]
    print("$ " + " ".join(cmd[:8]) + " ...", flush=True)
    subprocess.run(cmd, check=False)
else:
    print("no embeddings found — run W3-C to produce them")

In [ ]:
# --------------------------------------------------------------------------- #
# the deliverables, rendered here so the notebook is readable on its own
# --------------------------------------------------------------------------- #
for name in ("w3_tables.txt", "w3_diagnostics.txt"):
    p = TABLES / "tables" / name
    if not p.exists():
        continue
    print("\n" + "#" * 100)
    print(f"# {name}")
    print("#" * 100)
    print(p.read_text(encoding="utf-8"))

figs = sorted((TABLES / "figures").glob("*.png")) if (TABLES / "figures").is_dir() else []
print(f"\nfigures: {[f.name for f in figs]}")
print(f"tables : {sorted(p.name for p in (TABLES / 'tables').glob('*.csv'))}")
print()
print("next: Output tab -> Create Dataset for the tables, or download them directly.")

## reading the output

`tables/w3_tables.txt` is the paper draft in plain text: T1 leakage audit, T2
headline, T3 inflation gap, T4 leak decomposition, T5 per-seed spread, T6
parameter ladder, T7 domain generalisation, T8 cross-corpus, T9 label-shuffling.

`tables/all_runs.csv` is every run row, the raw material for anything not
tabulated here. `tables/bootstrap_ci.csv` holds the recording-level intervals,
and `tables/leak_scope.csv` the seen/unseen decomposition.

`figures/` holds F2 (protocol gap), F3 (leak decomposition) and F4 (scale curve).
F1 — the three-level leakage ladder — is drawn by hand from the frozen leak
audit, since it is a diagram of the dataset rather than of a run.